In [1]:
print("HEllo world")

HEllo world


## Docs, References
https://reference.langchain.com/python/langchain-core/runnables/base/Runnable

In [1]:
from helpers.common import (
  BASE_URL,
  DEEPSEEK_MODEL,
  QWEN_MODEL,
  langfuse,
  langfuse_handler,
  together_ai_llm
)



In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser



In [23]:
llm = ChatOllama(
  model=QWEN_MODEL,
  base_url=BASE_URL,
  validate_model_on_init=True
)

In [3]:
system_prompt = SystemMessagePromptTemplate.from_template("You are a {level} teacher. Answer in short points.")
human_prompt = HumanMessagePromptTemplate.from_template("Tell me about {topic} in {number} points.")
chat_prompt = ChatPromptTemplate([system_prompt, human_prompt])
chat_prompt

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])

## Without Chain

In [ ]:
# final_chat_prompt = chat_prompt.invoke({
#   'language': "hindi"
# })

# final_chat_prompt
# async for message in llm.astream(
#   final_chat_prompt,
#   config={"callbacks": [langfuse_handler], "run_name": "ollama-translator"},
# ):
#   print(message.content, end="")

ChatPromptValue(messages=[SystemMessage(content='You are a hindi translator. Translate the following into hindi but write in english language', additional_kwargs={}, response_metadata={}), HumanMessage(content='Good Morning People!', additional_kwargs={}, response_metadata={})])

# With Chain
### Sequential

In [4]:
chain = chat_prompt | together_ai_llm
chain

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])
| ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x79bc8b064ec0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x79bc8b065940>, root_client=<openai.OpenAI object at 0x79bc8b589be0>, root_async_client=<openai.AsyncOpenAI object at 0x79bc8b0656a0>, model_name='Qwen/Qwen3-Coder-480B-A35B-Instruct-FP8', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.t

In [8]:
for message in chain.stream({
  "level": "school",
  "topic": "Earth",
  "number": "2",

},
  config={"callbacks": [langfuse_handler], "run_name": "school-teacher"},
):
  print(message.content, end="")


1. **Habitable planet** – Earth is the only known planet with liquid water, a breathable atmosphere, and a magnetic field that protect life.  
2. **Diverse ecosystems** – It hosts deserts, forests, oceans, and mountains, each with unique plants, animals, and climates that interconnect through food webs and cycles.

# Chain Sequential | StrOutputParser

In [4]:
chain_1 = chat_prompt | together_ai_llm | StrOutputParser()
chain_1

ChatPromptTemplate(input_variables=['level', 'number', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['level'], input_types={}, partial_variables={}, template='You are a {level} teacher. Answer in short points.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['number', 'topic'], input_types={}, partial_variables={}, template='Tell me about {topic} in {number} points.'), additional_kwargs={})])
| ChatOpenAI(callbacks=[<langfuse.langchain.CallbackHandler.LangchainCallbackHandler object at 0x770c6cb2d160>], profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x770c62b3c1a0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x770c62b3c6e0>, root_client=<openai.OpenAI object at 0x770c6cb2d550>, root_async_client=<openai.AsyncOpenAI object at 0x770c62b3c440>, model_name='OpenAI/gpt-oss-20B', model_kw

In [6]:
# for message in chain_2.stream({
#   "level": "school",
#   "topic": "Earth",
#   "number": "2",

# },
#   config={"callbacks": [langfuse_handler], "run_name": "school-teacher"},
# ):
#   print(message, end="")
response = chain_1.invoke({
  "level": "school",
  "topic": "Earth",
  "number": "2",
})
response

'1. **Earth is a life‑supporting planet** – it has a breathable atmosphere, liquid water, and a magnetic field that protects us from cosmic radiation.\n\n2. **Its climate is shaped by its axis tilt and oceans** – this gives us seasons, drives weather systems, and allows diverse ecosystems across the globe.'

##  Chain Multiple Runnalbes
### One after the other

###### Example: You get the output from ek LLM. Woh output you send it to the LLM again to evaluate

In [ ]:
## Analysis Code
analysis_prompt = ChatPromptTemplate.from_template("""
Analyze the following text: {response}. You need to tell how how difficult it is to understand. Answer in one sentence only.
""")


chain_2 = analysis_prompt | together_ai_llm | StrOutputParser()
chain_2.invoke({
  'response': response
},
  config={
  "callbacks": [langfuse_handler], 
  "run_name": "analysis-chain",
    "metadata": {"langfuse_tags": ["open-ai"]},
  },
)


## Combining Chains

In [7]:
composed_chain = {'response': chain_1} | analysis_prompt | together_ai_llm | StrOutputParser()
output = composed_chain.invoke({
  "level": "school",
  "topic": "Earth",
  "number": "2",
},
  config={"run_name": "composed-chain (Topic+Analysis)",    "metadata": {"langfuse_tags": ["open-ai"]},},

)

In [13]:
output

'The passage is clear and straightforward, requiring only basic scientific vocabulary and a general understanding of Earth science, making it easy to comprehend.'

In [7]:
langfuse.flush()